In [ ]:
from datasets import load_dataset

# datasets 5.x 必须写 "命名空间/仓库名" 完整形式
mrpc = load_dataset("nyu-mll/glue", "mrpc")
print(mrpc)
print("---- 第一条训练样本 ----")
print(mrpc["train"][0])

In [2]:
import time

from datasets import load_dataset

# C4 因 gated 授权 + 镜像分页缺陷不可达，改用已验证可用的维基百科流式
wiki_stream = load_dataset("wikimedia/wikipedia", "20231101.en", streaming=True)

n = 0
start = time.perf_counter()   # 高精度计时器
for row in wiki_stream["train"]:
    n += 1
    if n >= 100:              # 只取 100 条看速度
        break
elapsed = time.perf_counter() - start

print(f"流式读取 {n} 条，用时 {elapsed:.2f} 秒，平均 {n / elapsed:.1f} 条/秒")
print("---- 最后一条的标题 ----")
print(row["title"])

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

流式读取 100 条，用时 8.87 秒，平均 11.3 条/秒
---- 最后一条的标题 ----
Agnostida


In [ ]:
import os

print("HF_ENDPOINT =", os.environ.get("HF_ENDPOINT"))

In [ ]:
import os

print("os 视角：", os.environ.get("HF_ENDPOINT"))

from huggingface_hub import HfApi

print("huggingface_hub 视角：", HfApi().endpoint)

In [3]:
import os

from datasets import load_dataset

# 命中缓存，秒回
mrpc = load_dataset("nyu-mll/glue", "mrpc")

os.makedirs("data", exist_ok=True)   # 没有 data 文件夹就建一个

ds = mrpc["train"]
ds.to_csv("data/mrpc.csv", index=False)
ds.to_json("data/mrpc.json")
ds.to_parquet("data/mrpc.parquet")

for fname in sorted(os.listdir("data")):
    size_kb = os.path.getsize(os.path.join("data", fname)) / 1024
    print(f"{fname:15s} {size_kb:8.1f} KB")

Creating CSV from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

mrpc.csv           893.8 KB
mrpc.json         1041.9 KB
mrpc.parquet       608.7 KB


In [ ]:
from datasets import DatasetDict, load_dataset

imdb_train = load_dataset("stanfordnlp/imdb", split="train")   # 同样命中缓存

# 第一步：70% train / 30% 临时池
split_70_30 = imdb_train.train_test_split(test_size=0.30, seed=42)
# 第二步：30% 对半 -> 15% val + 15% test（30% × 50% = 15%）
split_15_15 = split_70_30["test"].train_test_split(test_size=0.50, seed=42)

final = DatasetDict({
    "train": split_70_30["train"],
    "validation": split_15_15["train"],
    "test": split_15_15["test"],
})
print(final)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 17500
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 3750
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3750
    })
})
